# Marine Buoy Weather Forecasting
### Data Science for Business — Boe NOAA (1980–2023)

Questo notebook descrive passo per passo il processo di analisi e modellazione sviluppato per la previsione di variabili meteo-marine e l'analisi di similarità tra boe NOAA.

**Target:** `WVHT` (altezza significativa delle onde, m) e `WTMP` (temperatura dell'acqua, °C)  
**Boa principale (forecasting):** 42002 — Golfo del Messico (26°N, 93°W)  
**Boe aggiuntive (clustering):** 42001, 42039 (Golfo del Messico) + 46042 (Pacifico, Monterey)  
**Dataset:** [`Qdrant/NOAA-Buoy`](https://huggingface.co/datasets/Qdrant/NOAA-Buoy) su Hugging Face + NDBC

**Indice:**
1. Setup
2. Caricamento dati (`load.py`)
3. Pulizia dati (`clean.py`)
4. Analisi Esplorativa (EDA)
5. Feature engineering e split temporale (`features.py`)
6. Modelli di regressione e confronto (`models.py`)
7. AutoML con FLAML (`automl.py`)
8. Clustering delle boe (`clustering.py`)
9. Web Application
10. Conclusioni


## 1. Setup


In [ ]:
import sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

PROCESSED = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
CLUSTER   = PROCESSED / 'clustering'
RESULTS   = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

BLU, ARANCIO, VERDE = '#2563eb', '#ea580c', '#16a34a'
GRIGIO = '#93c5fd'
COLORI = [BLU, ARANCIO, VERDE, '#9333ea', '#ca8a04', '#1098ad']
print('ROOT:', ROOT)

**Cosa fa questo blocco:** importa le librerie necessarie e configura i percorsi delle cartelle del progetto. I grafici vengono salvati nella cartella `results/`.


---
## 2. Caricamento dei dati (`load.py`)

Il dataset principale è [`Qdrant/NOAA-Buoy`](https://huggingface.co/datasets/Qdrant/NOAA-Buoy) su Hugging Face: misurazioni orarie della boa NOAA **42002** nel Golfo del Messico dal **1980 al 2023**.

**Problema riscontrato:** la funzione standard `load_dataset('Qdrant/NOAA-Buoy')` fallisce con `DatasetGenerationCastError` perché i file CSV mensili hanno colonne inconsistenti tra loro.

**Soluzione adottata in `load.py`:** i file processati in formato `.parquet` vengono scaricati direttamente tramite `huggingface_hub`, bypassando `load_dataset`.

**Stazioni aggiuntive** scaricate da NDBC per l'analisi di clustering:
- 42001, 42039 — Golfo del Messico
- 46042 — Oceano Pacifico (Monterey, CA)

**Output:** `data/raw/buoys_all.csv` con tutte e 4 le stazioni.


In [ ]:
raw_f = ROOT / 'data' / 'raw' / 'buoys_all.csv'
if raw_f.exists():
    raw = pd.read_csv(raw_f, parse_dates=['timestamp'])
    print('Dataset grezzo (Qdrant/NOAA-Buoy + NDBC):')
    print(f'  Righe totali: {len(raw):,}')
    print(f'  Stazioni: {sorted(raw["station_id"].astype(str).unique())}')
    print(f'  Colonne: {list(raw.columns)}')
    for st, g in raw.groupby('station_id'):
        print(f'  {st}: {len(g):,} righe [{g["timestamp"].min().date()} → {g["timestamp"].max().date()}]')
else:
    print('File non trovato — esegui load.py prima')

**Cosa fa questo blocco:** carica il CSV grezzo e mostra righe per stazione e copertura temporale. La boa 42002 ha la storia più lunga (1980–2023); le stazioni NDBC coprono 2015–2023.


---
## 3. Pulizia dei dati (`clean.py`)

`clean.py` applica due operazioni su tutte le stazioni:

**Filtri di plausibilità fisica:** i valori fuori dai limiti fisicamente possibili vengono sostituiti con `NaN` (errori di sensore o valori sentinella). Esempi: WVHT > 30m, WTMP < -5°C o > 40°C, PRES fuori da 800–1100 hPa.

**Resampling orario** con `resample('1h').mean()`: crea una **griglia temporale regolare** con esattamente un'osservazione per ora. Questo è fondamentale: senza griglia regolare, il lag 1 non significa '1 ora fa' ma 'l'osservazione precedente', che potrebbe essere 30 minuti o 2 ore fa.

**Output:** `data/processed/buoys_clean.csv` (tutte le stazioni) + `buoy_42002_clean.csv` (boa primaria).


In [ ]:
df_all = pd.read_csv(PROCESSED / 'buoys_clean.csv', parse_dates=['timestamp'])
df_all['station_id'] = df_all['station_id'].astype(str)
print('Tutte le stazioni dopo la pulizia:')
for st, g in df_all.groupby('station_id'):
    miss_wvht = g['WVHT'].isna().mean()
    miss_wtmp = g['WTMP'].isna().mean()
    print(f'  {st}: {len(g):,} righe | WVHT missing={miss_wvht:.1%} | WTMP missing={miss_wtmp:.1%}')

df = pd.read_csv(PROCESSED / 'buoy_42002_clean.csv', parse_dates=['timestamp'])
print(f'\nBoa 42002: {len(df):,} righe, {df["timestamp"].dt.year.nunique()} anni')
display(df[['WVHT','WTMP','WSPD','PRES','ATMP']].describe().round(3))

**Cosa fa questo blocco:** mostra le percentuali di valori mancanti per stazione e le statistiche descrittive della boa 42002.


---
## 4. Analisi Esplorativa (EDA)

Prima di costruire i modelli, analizziamo la struttura dei dati per identificare:
- **Stagionalità:** pattern che si ripetono ciclicamente (estate/inverno)
- **Distribuzione:** forma, asimmetria, valori estremi
- **Autocorrelazione:** il valore attuale dipende dai valori passati?

Queste osservazioni guidano le scelte successive: la stagionalità motiva le feature di calendario; l'alta autocorrelazione spiega perché i lag sono le feature più predittive.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
df.set_index('timestamp')['WVHT'].dropna().plot(ax=axes[0], lw=0.3, color=BLU, alpha=0.8)
axes[0].set_title('WVHT — Altezza significativa delle onde (m) — Boa 42002 (1980–2023)', fontsize=12)
axes[0].set_ylabel('m'); axes[0].grid(alpha=0.3)
df.set_index('timestamp')['WTMP'].dropna().plot(ax=axes[1], lw=0.3, color=ARANCIO, alpha=0.8)
axes[1].set_title('WTMP — Temperatura dell\'acqua (°C) — Boa 42002 (1980–2023)', fontsize=12)
axes[1].set_ylabel('°C'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / '01_serie_storiche.png', dpi=120, bbox_inches='tight')
plt.show()

**Cosa fa questo blocco:** visualizza le serie storiche complete 1980–2023. La stagionalità annuale è chiaramente visibile: WVHT più alta in inverno, WTMP più alta in estate. Salvato in `results/01_serie_storiche.png`.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
df['WVHT'].dropna().plot(kind='hist', bins=60, ax=axes[0][0], color=BLU)
axes[0][0].set_title('WVHT — distribuzione'); axes[0][0].set_xlabel('m')
df['WTMP'].dropna().plot(kind='hist', bins=60, ax=axes[0][1], color=ARANCIO)
axes[0][1].set_title('WTMP — distribuzione'); axes[0][1].set_xlabel('°C')
df.assign(mese=df['timestamp'].dt.month).groupby('mese')['WVHT'].mean().plot(
    kind='bar', ax=axes[1][0], color=BLU)
axes[1][0].set_title('WVHT media per mese (onde più alte in inverno)')
axes[1][0].set_xlabel('Mese'); axes[1][0].set_ylabel('m')
df.assign(mese=df['timestamp'].dt.month).groupby('mese')['WTMP'].mean().plot(
    kind='bar', ax=axes[1][1], color=ARANCIO)
axes[1][1].set_title('WTMP media per mese (acqua più calda in estate)')
axes[1][1].set_xlabel('Mese'); axes[1][1].set_ylabel('°C')
plt.suptitle('EDA — Distribuzione e Stagionalità', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / '02_eda.png', dpi=120, bbox_inches='tight')
plt.show()

**Cosa fa questo blocco:** distribuzioni e medie mensili. La stagionalità confermata qui motiva l'uso di feature di calendario (sin/cos). Salvato in `results/02_eda.png`.


---
## 5. Feature Engineering e Split Temporale (`features.py`)

### Trasformazione in problema supervisionato
Per addestrare i modelli ML, la serie temporale viene convertita:
- **y** = valore del target all'ora `t+1` (quello da prevedere)
- **X** = tutto ciò che è noto fino all'ora `t` (nessuna informazione dal futuro)

### Feature costruite (senza data leakage)
- **Lag del target:** `WVHT_lag1`, `lag2`, `lag3`, `lag6`, `lag12`, `lag24` — sfruttano l'autocorrelazione
- **Rolling statistics:** `WVHT_roll3`, `roll6`, `roll24` — tendenza recente
- **Variabili esogene:** WSPD, PRES, ATMP, ecc. al tempo corrente + lag1
- **Calendario ciclico:** `hour_sin/cos` (ciclo giornaliero), `doy_sin/cos` (stagionalità annuale)

**Perché codifica ciclica?** L'ora 23 è vicina all'ora 0 ma numericamente distante. Sin e cos preservano questa continuità.

### Split temporale 70% / 15% / 15% — NON casuale
Lo shuffle casuale introdurrebbe **data leakage**: il modello vedrebbe dati futuri durante il training.
```
[========== TRAIN 70% ==========][=== VAL 15% ===][=== TEST 15% ===]
         1980 → ~2013                ~2013→2018        2018→2023
```
Il modello migliore viene scelto sulla **validation** — il test viene toccato UNA SOLA VOLTA alla fine.


In [ ]:
import joblib
for target in ['WVHT', 'WTMP']:
    train = pd.read_csv(PROCESSED / f'train_42002_{target}.csv', parse_dates=['timestamp'])
    val   = pd.read_csv(PROCESSED / f'val_42002_{target}.csv',   parse_dates=['timestamp'])
    test  = pd.read_csv(PROCESSED / f'test_42002_{target}.csv',  parse_dates=['timestamp'])
    feat  = [c for c in train.columns if c not in ('timestamp','y')]
    print(f'\n=== {target} — {len(feat)} feature ===')
    print(f'  Train: {len(train):>7,} righe [{train["timestamp"].min().date()} → {train["timestamp"].max().date()}]')
    print(f'  Val:   {len(val):>7,} righe [{val["timestamp"].min().date()} → {val["timestamp"].max().date()}]')
    print(f'  Test:  {len(test):>7,} righe [{test["timestamp"].min().date()} → {test["timestamp"].max().date()}]')
    print(f'  Prime feature: {feat[:6]}')

**Cosa fa questo blocco:** mostra dimensioni e intervalli temporali esatti dei tre split. Verifica che train < val < test senza sovrapposizioni.


---
## 6. Modelli di Regressione e Confronto (`models.py`)

Vengono addestrati e confrontati 4 approcci per ciascun target e per ciascuna boa.

| Modello | Tipo | Caratteristica principale |
|---|---|---|
| **Persistence** | Baseline naive | ŷ(t+1) = y(t). Nessun training. Riferimento minimo. |
| **Ridge** | Lineare + L2 | Previene overfitting con feature correlate (lag). Richiede StandardScaler. |
| **Random Forest** | Bagging | Molti alberi su sottocampioni casuali, media delle previsioni. Riduce varianza. |
| **Gradient Boosting** | Boosting | Alberi sequenziali sui residui. Riduce il bias. Generalmente più accurato. |

**Selezione:** migliore modello scelto su **validation RMSE** — il test set non viene mai usato per decisioni.

**Nota su boe 42001 e 46042:** R² negativo sul test set indica che il modello generalizza peggio della media semplice. Causa: dati insufficienti (solo 2015–2023, ~9 anni vs 40+ anni della 42002). È un risultato reale che conferma il principio: più dati storici = modelli migliori.


In [ ]:
metrics_f = MODELS / 'metrics_42002.json'
if not metrics_f.exists(): metrics_f = MODELS / 'metrics.json'
if not metrics_f.exists():
    print('Esegui models.py prima')
else:
    m = json.loads(metrics_f.read_text())
    for target, info in m.items():
        print(f'\n=== {target} === Migliore su val RMSE: {info["best"].upper()}')
        rows = [{'modello':n,
            'val MAE':v['val']['MAE'],'val RMSE':v['val']['RMSE'],'val R²':v['val']['R2'],
            'test MAE':v['test']['MAE'],'test RMSE':v['test']['RMSE'],'test R²':v['test']['R2']}
            for n,v in info['metrics'].items()]
        display(pd.DataFrame(rows).sort_values('val RMSE').reset_index(drop=True))
        p = info['metrics']['persistence']['test']['RMSE']
        b = info['metrics'][info['best']]['test']['RMSE']
        print(f'  → Miglioramento rispetto alla persistence: {(p-b)/p*100:.1f}%')

**Cosa fa questo blocco:** carica le metriche della boa 42002 e mostra la tabella comparativa ordinata per validation RMSE. Il miglioramento % misura quanto il modello batte la persistence.


In [ ]:
if metrics_f.exists():
    m = json.loads(metrics_f.read_text())
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for row, target in enumerate(['WVHT','WTMP']):
        info = m[target]
        nomi = list(info['metrics'].keys())
        for col, (split, met) in enumerate([('val','RMSE'),('val','R2'),('test','RMSE')]):
            vals   = [info['metrics'][n][split][met] for n in nomi]
            colori = [BLU if n==info['best'] else GRIGIO for n in nomi]
            axes[row][col].bar(nomi, vals, color=colori)
            axes[row][col].set_title(f'{target} — {split} {met}', fontsize=10)
            axes[row][col].tick_params(axis='x', rotation=25, labelsize=8)
            if met=='R2': axes[row][col].set_ylim(0,1)
            axes[row][col].grid(axis='y', alpha=0.3)
    plt.suptitle('Confronto modelli — blu = migliore su validation RMSE (boa 42002)', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '03_confronto_modelli.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** grafici a barre che confrontano tutti i modelli per entrambi i target. La barra blu è il modello selezionato. Salvato in `results/03_confronto_modelli.png`.


In [ ]:
if metrics_f.exists():
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    for i, target in enumerate(['WVHT','WTMP']):
        test_f   = PROCESSED / f'test_42002_{target}.csv'
        bundle_f = MODELS / f'best_42002_{target}.joblib'
        if not bundle_f.exists(): bundle_f = MODELS / f'best_{target}.joblib'
        if not test_f.exists() or not bundle_f.exists(): continue
        test   = pd.read_csv(test_f, parse_dates=['timestamp'])
        bundle = joblib.load(bundle_f)
        X = test[bundle['features']].values.astype(float)
        if bundle['scaler']: X = bundle['scaler'].transform(X)
        yhat  = bundle['model'].predict(X)
        ytrue = test['y'].values
        pers  = test[f'{target}_t0'].values
        sl = slice(-500, None)
        axes[i].plot(test['timestamp'].iloc[sl], ytrue[sl], label='valore reale', lw=1.5, color='#1e293b')
        axes[i].plot(test['timestamp'].iloc[sl], yhat[sl], label=f'predetto ({bundle["name"]})', lw=1, color=BLU, alpha=0.85)
        axes[i].plot(test['timestamp'].iloc[sl], pers[sl], label='persistence (baseline)', lw=0.8, color=ARANCIO, linestyle='--', alpha=0.7)
        axes[i].set_title(f'{target} — reale vs predetto vs persistence (ultimi 500 punti test)', fontsize=11)
        axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS / '04_previsione_vs_reale.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** grafico reale vs predetto vs persistence sugli ultimi 500 punti del test set. Il modello (blu) deve seguire il nero più da vicino della persistence (arancio). Salvato in `results/04_previsione_vs_reale.png`.


---
## 7. AutoML con FLAML (`automl.py`)

**FLAML** (Fast and Lightweight AutoML, Microsoft): cerca automaticamente il modello migliore e i suoi iperparametri entro un budget di tempo. Invece di configurare manualmente ogni algoritmo, FLAML esplora LightGBM, XGBoost, Random Forest, Ridge e altri, restituendo la configurazione ottimale.

**Configurazione utilizzata:**
```python
automl.fit(
    task='regression',    # problema di regressione
    metric='rmse',        # ottimizza l'RMSE
    time_budget=60,       # 60 secondi di ricerca
    eval_method='cv',
    split_type='time',    # CV temporale — nessun data leakage
    n_splits=5,
)
```

`split_type='time'` garantisce che in ogni fold di cross-validation il training preceda sempre la validation — coerente con lo split manuale.

**Risultati boa 42002:**
- WVHT → XGBoost (Test RMSE=0.0905, R²=0.9807)
- WTMP → LGBM (Test RMSE=0.0935, R²=0.9989)


In [ ]:
automl_f = MODELS / 'metrics_automl_42002.json'
if not automl_f.exists(): automl_f = MODELS / 'metrics_automl.json'
if not automl_f.exists():
    print('Esegui automl.py prima')
else:
    a = json.loads(automl_f.read_text())
    print('Risultati AutoML — FLAML (budget 60 secondi, boa 42002):')
    for target, info in a.items():
        print(f'\n  {target}: algoritmo scelto = {info["best_estimator"]}')
        print(f'    Val  → RMSE={info["metrics"]["val"]["RMSE"]}  R²={info["metrics"]["val"]["R2"]}')
        print(f'    Test → RMSE={info["metrics"]["test"]["RMSE"]}  R²={info["metrics"]["test"]["R2"]}')

**Cosa fa questo blocco:** mostra l'algoritmo scelto automaticamente da FLAML e le sue metriche su validation e test per la boa 42002.


In [ ]:
if metrics_f.exists() and automl_f.exists():
    m = json.loads(metrics_f.read_text())
    a = json.loads(automl_f.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, target in enumerate(['WVHT','WTMP']):
        info  = m[target]
        nomi  = list(info['metrics'].keys()) + [f'AutoML\n({a[target]["best_estimator"]})']
        rmses = ([info['metrics'][n]['test']['RMSE'] for n in info['metrics']]
                 + [a[target]['metrics']['test']['RMSE']])
        col = [ARANCIO if 'AutoML' in n else (BLU if n.split('\n')[0].strip()==info['best'] else GRIGIO) for n in nomi]
        axes[i].bar(nomi, rmses, color=col)
        axes[i].set_title(f'{target} — Test RMSE\nblu=migliore sklearn, arancio=AutoML')
        axes[i].tick_params(axis='x', rotation=20, labelsize=8)
        axes[i].grid(axis='y', alpha=0.3)
    plt.suptitle('Confronto completo — modelli classici vs AutoML (FLAML, 60s) — Boa 42002', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '05_confronto_automl.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** confronto completo tra modelli sklearn e AutoML in un unico grafico. Salvato in `results/05_confronto_automl.png`.


---
## 8. Clustering delle Boe (`clustering.py`)

L'analisi di clustering risponde alla domanda: **quali boe hanno comportamenti meteorologici simili e come cambiano nel tempo?**

Ogni boa/periodo è descritto da: **media + deviazione standard di WVHT, WTMP, WSPD, PRES, ATMP** (10 feature totali).

### Pipeline
1. **StandardScaler** — normalizza le feature (KMeans usa distanze euclidee, sensibile alle scale)
2. **KMeans** — prova k da 2 a 8
3. **Silhouette Score** — sceglie automaticamente il k migliore (più alto = cluster meglio separati)
4. **Elbow Plot** — mostra l'inertia al variare di k (metodo alternativo)
5. **PCA a 2 componenti** — riduce le 10 feature a 2D per la visualizzazione

### Clustering Statico (profili complessivi)
Ogni punto = **una boa**, descritta dal suo profilo medio sull'intero periodo disponibile.
Mostra quali boe hanno condizioni meteo-marine simili nel lungo termine.

**Risultato (k=2, silhouette=0.236):**
- Cluster 0: 42001, 42002, 42039 — boe del Golfo del Messico, profilo simile
- Cluster 1: 46042 — Pacifico, regime completamente diverso (onde più alte, acqua più fredda)

### Clustering Dinamico (profili annuali)
Ogni punto = **(boa, anno)**, descritto dalle condizioni medie di quell'anno.
Mostra come il regime di ogni boa cambia anno per anno.

**Risultato (k=2, silhouette=0.631, 49 punti):**
- Boa 42002 sempre in cluster 0 per tutti i 43 anni — regime stabile
- Boa 46042 sempre in cluster 1 — sempre separata dalle boe del Golfo


In [ ]:
static_f = CLUSTER / 'static.json'
if not static_f.exists():
    print('Esegui clustering.py prima')
else:
    s   = json.loads(static_f.read_text())
    tab = pd.read_csv(CLUSTER / 'static.csv')
    print('Clustering Statico (profili complessivi per boa):')
    print(f'  k={s["k"]}  silhouette={s["silhouette"]}  stazioni={s["n_stations"]}')
    for c, stazioni in s['clusters'].items():
        print(f'  Cluster {c}: {stazioni}')

**Cosa fa questo blocco:** mostra i risultati del clustering statico — quali boe sono nello stesso cluster e quali sono separate.


In [ ]:
if static_f.exists():
    s   = json.loads(static_f.read_text())
    tab = pd.read_csv(CLUSTER / 'static.csv')
    fig = plt.figure(figsize=(15, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig)

    ax0 = fig.add_subplot(gs[0])
    for c in sorted(tab['cluster'].unique()):
        sub = tab[tab['cluster']==c]
        ax0.scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}',
                    s=120, color=COLORI[c % len(COLORI)], zorder=3)
        for _, r in sub.iterrows():
            ax0.annotate(str(r['station_id']), (r['pca_x'], r['pca_y']),
                         fontsize=9, xytext=(5,5), textcoords='offset points')
    ax0.set_xlabel('PCA 1'); ax0.set_ylabel('PCA 2')
    ax0.set_title(f'PCA 2D — ogni punto = 1 boa\nk={s["k"]} sil={s["silhouette"]}')
    ax0.legend(); ax0.grid(alpha=0.3)

    ax1 = fig.add_subplot(gs[1])
    sil_k = sorted([int(k) for k in s['silhouette_by_k']])
    sil_v = [s['silhouette_by_k'][str(k)] for k in sil_k]
    ax1.bar([f'k={k}' for k in sil_k], sil_v,
            color=[BLU if k==s['k'] else GRIGIO for k in sil_k])
    ax1.set_title('Silhouette Score per k\npiù alto = cluster più separati\nblu = k scelto')
    ax1.set_ylim(0, max(sil_v)*1.2 if sil_v else 1); ax1.grid(axis='y', alpha=0.3)

    ax2 = fig.add_subplot(gs[2])
    inertia_k = sorted([int(k) for k in s['inertia_by_k']])
    inertia_v = [s['inertia_by_k'][str(k)] for k in inertia_k]
    ax2.plot([f'k={k}' for k in inertia_k], inertia_v, 'o-', color=BLU, lw=2, markersize=8)
    ax2.set_title('Elbow Plot — Inertia per k\n(cerca il gomito nella curva)')
    ax2.grid(alpha=0.3)

    plt.suptitle('Clustering Statico — Profili per boa (ogni punto = 1 boa)', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '06_clustering_statico.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** tre grafici del clustering statico — scatter PCA 2D (ogni punto è una boa etichettata), silhouette per k, elbow plot. La separazione netta di 46042 (Pacifico) dalle boe del Golfo è chiaramente visibile. Salvato in `results/06_clustering_statico.png`.


In [ ]:
dyn_f = CLUSTER / 'dynamic.json'
if not dyn_f.exists():
    print('Esegui clustering.py prima')
else:
    d    = json.loads(dyn_f.read_text())
    dtab = pd.read_csv(CLUSTER / 'dynamic.csv')
    print('Clustering Dinamico (profili annuali per boa):')
    print(f'  k={d["k"]}  silhouette={d["silhouette"]}  punti={d["n_points"]}')
    print('  Sequenze di regime:')
    for st, seq in d['sequences'].items():
        print(f'    {st}: {" → ".join(map(str, seq["clusters"]))} ({seq["changes"]} cambi)')

**Cosa fa questo blocco:** mostra le sequenze di regime per ogni boa anno per anno. Nessun cambio di cluster = regime stabile nel tempo.


In [ ]:
if dyn_f.exists():
    d    = json.loads(dyn_f.read_text())
    dtab = pd.read_csv(CLUSTER / 'dynamic.csv')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for c in sorted(dtab['cluster'].unique()):
        sub = dtab[dtab['cluster']==c]
        axes[0].scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}',
                        color=COLORI[c % len(COLORI)], s=60, zorder=3)
        for _, r in sub.iterrows():
            axes[0].annotate(f"{r['station_id']}\n{int(r['year'])}",
                             (r['pca_x'], r['pca_y']),
                             fontsize=6, xytext=(3,3), textcoords='offset points')
    axes[0].set_xlabel('PCA 1'); axes[0].set_ylabel('PCA 2')
    axes[0].set_title(f'PCA 2D — ogni punto = boa in un anno\nk={d["k"]} sil={d["silhouette"]}')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    stazioni_lista = sorted(d['sequences'].keys())
    for idx, st in enumerate(stazioni_lista):
        seq = d['sequences'][st]
        for yr, cl in zip(seq['years'], seq['clusters']):
            axes[1].bar(yr, 1, bottom=idx, color=COLORI[cl % len(COLORI)], width=0.8, alpha=0.85)
    axes[1].set_yticks(range(len(stazioni_lista)))
    axes[1].set_yticklabels(stazioni_lista)
    axes[1].set_title('Regime per anno per boa\n(colore = cluster)')
    axes[1].set_xlabel('Anno')
    handles = [mpatches.Patch(color=COLORI[c], label=f'Cluster {c}') for c in range(d['k'])]
    axes[1].legend(handles=handles, loc='upper right', fontsize=8)
    axes[1].grid(axis='x', alpha=0.2)

    plt.suptitle('Clustering Dinamico — Come i regimi variano anno per anno', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '07_clustering_dinamico.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** scatter PCA (boa × anno) e grafico a barre che mostra il cluster di ogni boa per ogni anno. I colori diversi indicano regimi diversi. Salvato in `results/07_clustering_dinamico.png`.


---
## 9. Web Application

Il progetto espone un'applicazione web con due componenti FastAPI in un singolo container Docker gestito da **supervisord**:

### Backend — API REST (porta 8080)
Carica i bundle `.joblib`, esegue l'inferenza, legge i risultati del clustering. Usa **Pydantic** per la validazione dei dati.

| Endpoint | Descrizione |
|---|---|
| `GET /api/health` | Stato del servizio |
| `GET /api/stations` | Lista delle boe disponibili |
| `GET /api/predict` | Previsione WVHT + WTMP a t+1h |
| `GET /api/comparison` | MAE/RMSE/R² di tutti i modelli per tutte le boe |
| `GET /api/clusters/static` | Clustering statico |
| `GET /api/clusters/dynamic` | Clustering dinamico annuale |

### Frontend — Interfaccia Web (porta 8000)
Chiama il backend via `httpx`, renderizza le pagine con **Jinja2** e grafici interattivi con **Chart.js**.

| Pagina | Contenuto |
|---|---|
| `/` | Dashboard: descrizione progetto + lista boe |
| `/prediction` | Dropdown boa + grafico storico + punto di previsione |
| `/comparison` | Tabella metriche tutti i modelli per tutte le boe |
| `/clustering` | PCA scatter + silhouette + elbow + regimi annuali |

### Avvio con Docker
```bash
# Pull ed esecuzione diretta (immagine già con modelli inclusi)
docker pull ghcr.io/brusca01/marine-buoy-forecasting:latest
docker run --rm -p 8000:8000 -p 8080:8080 ghcr.io/brusca01/marine-buoy-forecasting:latest
```
Frontend → http://localhost:8000 | Backend → http://localhost:8080/docs


---
## 10. Conclusioni

### Cosa è stato fatto passo per passo
1. **Scaricato** i dati della boa 42002 da `Qdrant/NOAA-Buoy` (HF) bypassando il bug di `load_dataset` + boe NDBC aggiuntive
2. **Pulito** i dati: filtri fisici + resampling orario per griglia regolare
3. **Analizzato** stagionalità, distribuzioni e autocorrelazione (EDA)
4. **Costruito** lag features, rolling stats, calendario ciclico — nessun data leakage
5. **Addestrato** persistence, Ridge, Random Forest, Gradient Boosting con split 70/15/15 temporale
6. **Confrontato** con AutoML (FLAML, 60s, CV temporale)
7. **Clusterizzato** le boe per profilo complessivo (statico) e per anno (dinamico)
8. **Deployato** tutto in FastAPI backend + frontend in Docker

### Risultati principali
- **WVHT** sulla boa 42002: R² = 0.9809 con Random Forest — alta prevedibilità grazie all'autocorrelazione
- **WTMP** sulla boa 42002: R² = 0.9991 con Ridge — variazione lentissima, quasi deterministicamente prevedibile
- **R² negativo** su 42001 e 46042: dati insufficienti (9 anni vs 40+) — conferma empirica che la quantità di dati è determinante
- **Clustering statico**: separa nettamente la boa pacifica (46042) dalle boe del Golfo — diverso regime meteo-marino
- **Clustering dinamico**: regimi stabili nel tempo per tutte le boe — nessun cambio di cluster rilevato
- **AutoML**: trova XGBoost/LightGBM automaticamente, competitivo con i modelli configurati manualmente

**Dataset:** https://huggingface.co/datasets/Qdrant/NOAA-Buoy
